<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Machine Learning Fundamentals** · Week 1 · Your First Models with scikit-learn · Lesson 4 of 24

# Logistic Regression: Predicting Yes or No

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/machine-learning-fundamentals).

**Learning objective:** Train a logistic regression classifier, turn its output into probabilities and decisions, and explain why accuracy alone can mislead.

## The problem: will the loan be repaid?

Mobile-money lenders in Rwanda and across Africa give small loans in seconds, using data instead of paperwork. The model must answer a **yes/no** question: will this borrower **default** (fail to repay)? That is **classification**.

Run the loans cell. It creates 2,000 past borrowers. `defaulted = 1` means the loan was not repaid.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(11)
n = 2000
income = rng.lognormal(12.2, 0.45, n).round(-3)
loan = rng.uniform(20_000, 300_000, n).round(-3)
months = rng.integers(1, 61, n)
late = rng.poisson(0.6, n)
salary = rng.random(n) < 0.55
z = -3.2 + 2.5 * (loan / income) - 0.03 * months + 0.8 * late - 0.9 * salary
default = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)
loans = pd.DataFrame({"income_rwf": income, "loan_rwf": loan, "months_customer": months,
                      "late_payments": late, "salaried": salary.astype(int),
                      "defaulted": default})
print(loans.shape, "default rate:", round(loans["defaulted"].mean(), 3))
loans.head()

About **21%** of borrowers defaulted.

## From a line to a probability

Linear regression can give any number, like −3 or 250. For yes/no we want a **probability** between 0 and 1. Logistic regression computes a straight-line score, then squeezes it through the **sigmoid** (S-shaped) curve:

> probability = 1 ÷ (1 + e^(−score))

A very negative score gives a probability near 0, a score of 0 gives 0.5, a large score gives nearly 1. Despite its name, logistic regression is a **classification** model.

## Train it

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = loans.drop(columns="defaulted")
y = loans["defaulted"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

clf = LogisticRegression().fit(X_train_s, y_train)
print("Accuracy:", accuracy_score(y_test, clf.predict(X_test_s)))

`stratify=y` keeps the same share of defaulters (21%) in both the training and test sets — important whenever one class is smaller.

You should get about **0.84** accuracy.

## Is 84% good? Check the baseline

In [ ]:
from sklearn.dummy import DummyClassifier

lazy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Baseline accuracy:", accuracy_score(y_test, lazy.predict(X_test)))

A "model" that says **nobody ever defaults** scores about **0.79** — and is useless, because it never catches a single bad loan. Our model is better, but only by 5 points. In Week 3 you will learn measures (precision, recall) that show what really matters.

## Probabilities, not just labels

In [ ]:
new = pd.DataFrame({
    "income_rwf":      [150_000, 400_000],
    "loan_rwf":        [250_000,  50_000],
    "months_customer": [3, 40],
    "late_payments":   [2, 0],
    "salaried":        [0, 1],
})
print(clf.predict_proba(scaler.transform(new))[:, 1].round(3))

The first applicant (low income, big loan, new customer, two late payments) gets a default probability of about **0.92**; the second about **0.001**. `predict()` simply says "1" when the probability is above **0.5** — but a lender can choose another **threshold**, for example refuse above 0.3 (Lesson 15).

## Reading the coefficients

In [ ]:
print(pd.Series(clf.coef_[0], index=X.columns).round(2))

Because the features are standardised, the sizes can be compared:

- **income_rwf** is negative: higher income → lower risk.
- **loan_rwf** and **late_payments** are positive: bigger loans and past lateness → higher risk.
- **months_customer** and **salaried** are negative: loyal and salaried customers are safer.

These directions make business sense — a good check that the model learned something real.

## More than two classes

Logistic regression also handles several classes (for example: low, medium, high risk). scikit-learn does this automatically when y has more than two values.

## Key terms

- **Classification** — predicting a category.
- **Logistic regression** — a linear score passed through the sigmoid to give a probability.
- **Sigmoid** — S-shaped curve that turns any number into 0–1.
- **predict_proba** — the predicted probability of each class.
- **Threshold** — the probability above which we say "yes" (0.5 by default).
- **Stratify** — keep class shares the same in train and test.

**Takeaway:** Logistic regression gives a probability for each case; accuracy must always be compared with the "most frequent class" baseline.

## Try it yourself

1. Train without scaling (use `LogisticRegression(max_iter=1000)`). Is the accuracy very different? Are the coefficients easier or harder to compare?
2. Find the five test borrowers with the highest predicted default probability. Look at their features — do they make sense?
3. Count how many test borrowers have a probability above 0.3, and how many above 0.5.
4. Explain to a loan officer, in three sentences, why 79% accuracy can be useless.

---
Finished? Go back to [Machine Learning Fundamentals on WAMUGII](https://wamugii.kento.rw/courses/machine-learning-fundamentals), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*